In [1]:
pip install neo4j pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
# neo4j의 scipama 데이터베이스 -> python 연결
from neo4j import GraphDatabase
import pandas as pd


uri = "bolt://localhost:7687"
user = "neo4j"
password = "gustjs21@"
database='sicpama'
driver = GraphDatabase.driver(uri, auth=(user, password))

def run_cypher(query):
    with driver.session(database=database) as session:
        result = session.run(query)
        return pd.DataFrame([r.data() for r in result])



In [6]:
# 실행 예
df = run_cypher("MATCH (c:Customer) RETURN c.fullName LIMIT 5")
print(df)

  c.fullName
0      G***t
1        문*라
2        임*민
3        조*영
4        강*윤


In [9]:
# 한 식당에 3번 이상 방문한 사람
cypher_query = """
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:PLACED_AT]->(s:Store)
WITH c.fullName AS customer, s.displayName AS store, count(o) AS visit_count
WHERE visit_count >= 3
RETURN customer, store, visit_count
ORDER BY visit_count DESC
"""

df = run_cypher(cypher_query)
print(df)


         customer              store  visit_count
0           G***t         장도뚝배기 동성로점        21725
1           G***t          88식당(영대점)        15043
2           G***t             유방녕의 웍         5776
3           G***t  PERF TEST [Chris]         2721
4           G***t           카페베네 신촌점         2488
...           ...                ...          ...
4124          유*훈           카페베네 신촌점            3
4125          천*섭           카페베네 신촌점            3
4126  P*********H           카페베네 신촌점            3
4127          박*아           카페베네 신촌점            3
4128          이*주           카페베네 신촌점            3

[4129 rows x 3 columns]


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import koreanize_matplotlib

In [ ]:
MATCH (c1:Customer)-[:PLACED]->(o:Order)<-[:PLACED]-(c2:Customer)
WHERE c1 <> c2
WITH c1, COUNT(DISTINCT o) AS shared_orders
WHERE shared_orders >= 2
RETURN c1.name AS inviter, shared_orders
ORDER BY shared_orders DESC
